# 11 — Batch Processing: Process Documents at Scale

**Time**: ~10 minutes · **Pattern**: Concurrent processing with retry logic

---

## Insurance Scenario

An insurance company processes **thousands of documents daily** — claim forms, invoices, receipts, and ID documents. You need to:

- Process multiple documents concurrently
- Handle rate limiting (429 errors) gracefully
- Track processing status and errors
- Respect service quotas (15 TPS default)

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
import time
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
from azure.core.exceptions import HttpResponseError
import pandas as pd

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Service Quotas to Know

| Quota | Free (F0) | Standard (S0) |
|---|---|---|
| Analyze requests/sec | 1 TPS | 15 TPS (adjustable) |
| Max document size | 4 MB | 500 MB |
| Max pages per doc | 2 | 2,000 |
| Get results/sec | 1 TPS | 50 TPS |

## Pattern 1: Sequential Processing with Retry

The simplest approach — process one document at a time with exponential backoff on errors.

In [ ]:
def analyze_with_retry(client, model_id, document_url, max_retries=3):
    """
    Analyze a document with exponential backoff retry.
    Handles 429 (rate limit) and transient errors.
    """
    for attempt in range(max_retries):
        try:
            poller = client.begin_analyze_document(
                model_id,
                AnalyzeDocumentRequest(url_source=document_url)
            )
            result = poller.result()
            return {"status": "success", "result": result, "attempts": attempt + 1}
        except HttpResponseError as e:
            if e.status_code == 429:  # Rate limited
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"  ⏳ Rate limited. Waiting {wait_time}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            elif e.status_code in (500, 502, 503):  # Transient server errors
                wait_time = 2 ** attempt
                print(f"  ⏳ Server error {e.status_code}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                return {"status": "error", "error": str(e), "attempts": attempt + 1}
    
    return {"status": "failed", "error": "Max retries exceeded", "attempts": max_retries}

print("analyze_with_retry() function ready.")

In [ ]:
# Simulate a batch of insurance documents
documents = [
    {"id": "INV-001", "type": "invoice", "model": "prebuilt-invoice",
     "url": "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"},
    {"id": "REC-001", "type": "receipt", "model": "prebuilt-receipt",
     "url": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/documentintelligence/azure-ai-documentintelligence/samples/sample_forms/receipt/contoso-receipt.png"},
    {"id": "DOC-001", "type": "layout", "model": "prebuilt-layout",
     "url": "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"},
]

print(f"Batch: {len(documents)} documents to process")
for doc in documents:
    print(f"  {doc['id']:10s} | {doc['type']:10s} | model: {doc['model']}")

In [ ]:
# Process the batch sequentially
batch_results = []

print("Processing batch...")
print("=" * 60)

for doc in documents:
    print(f"\n📄 Processing {doc['id']} ({doc['type']})...")
    start = time.time()
    
    result = analyze_with_retry(client, doc["model"], doc["url"])
    elapsed = time.time() - start
    
    batch_results.append({
        "Document ID": doc["id"],
        "Type": doc["type"],
        "Status": result["status"],
        "Attempts": result["attempts"],
        "Time (s)": f"{elapsed:.1f}",
        "Pages": len(result["result"].pages) if result["status"] == "success" else "-",
    })
    
    if result["status"] == "success":
        print(f"  ✅ Success in {elapsed:.1f}s ({result['attempts']} attempt(s))")
    else:
        print(f"  ❌ {result['status']}: {result.get('error', 'Unknown')}")

print("\n" + "=" * 60)
print("BATCH SUMMARY:")
df = pd.DataFrame(batch_results)
print(df.to_string(index=False))

## Pattern 2: Concurrent Submission with Polling

For higher throughput, submit all documents first (fire-and-forget), then poll for results. This is more efficient because analysis happens in parallel on the server side.

In [ ]:
def batch_analyze_concurrent(client, documents, delay_between_submissions=0.5):
    """
    Submit all documents concurrently, then collect results.
    
    Args:
        client: DocumentIntelligenceClient
        documents: list of {id, model, url} dicts
        delay_between_submissions: seconds between submissions (to avoid rate limiting)
    """
    # Phase 1: Submit all documents
    pollers = {}
    print("Phase 1: Submitting documents...")
    for doc in documents:
        try:
            poller = client.begin_analyze_document(
                doc["model"],
                AnalyzeDocumentRequest(url_source=doc["url"])
            )
            pollers[doc["id"]] = {"poller": poller, "doc": doc}
            print(f"  ✅ Submitted {doc['id']}")
            time.sleep(delay_between_submissions)  # Avoid burst rate limiting
        except HttpResponseError as e:
            print(f"  ❌ Failed to submit {doc['id']}: {e.message}")
    
    # Phase 2: Collect results
    print(f"\nPhase 2: Collecting results for {len(pollers)} documents...")
    results = {}
    for doc_id, item in pollers.items():
        try:
            result = item["poller"].result()
            results[doc_id] = {"status": "success", "result": result}
            pages = len(result.pages)
            print(f"  ✅ {doc_id}: {pages} page(s) analyzed")
        except Exception as e:
            results[doc_id] = {"status": "error", "error": str(e)}
            print(f"  ❌ {doc_id}: {str(e)[:80]}")
    
    return results

print("batch_analyze_concurrent() function ready.")

In [ ]:
# Run concurrent batch
start = time.time()
results = batch_analyze_concurrent(client, documents)
total_time = time.time() - start

success = sum(1 for r in results.values() if r["status"] == "success")
print(f"\n📊 Batch complete: {success}/{len(documents)} succeeded in {total_time:.1f}s")

## Pattern 3: Process Local Files from a Directory

In [ ]:
import glob

def process_directory(client, directory_path, model_id, file_pattern="*.pdf"):
    """
    Process all matching files in a directory.
    
    Args:
        directory_path: Path to directory containing documents
        model_id: Which model to use for analysis
        file_pattern: Glob pattern for files (e.g., '*.pdf', '*.png')
    """
    files = glob.glob(os.path.join(directory_path, file_pattern))
    print(f"Found {len(files)} files matching '{file_pattern}' in {directory_path}")
    
    results = []
    for filepath in files:
        filename = os.path.basename(filepath)
        print(f"  Processing {filename}...")
        
        try:
            with open(filepath, "rb") as f:
                poller = client.begin_analyze_document(model_id, body=f)
                result = poller.result()
            
            results.append({
                "file": filename,
                "status": "success",
                "pages": len(result.pages),
                "result": result,
            })
            print(f"    ✅ {len(result.pages)} page(s)")
        except Exception as e:
            results.append({
                "file": filename,
                "status": "error",
                "error": str(e),
            })
            print(f"    ❌ {str(e)[:60]}")
    
    return results

# Example usage (uncomment and update path):
# results = process_directory(client, "../sample-documents", "prebuilt-invoice", "*.pdf")

print("process_directory() function ready.")
print("Uncomment the example above and point to a folder with documents.")

## Best Practices for Production Batch Processing

1. **Respect rate limits**: Space submissions by 100-200ms to stay under 15 TPS
2. **Implement exponential backoff**: On 429 errors, wait 1s → 2s → 4s → 8s
3. **Use the GET polling wisely**: Don't poll more than once every 2 seconds per document
4. **Request TPS increases**: For high-volume production workloads, request a TPS increase via Azure Support
5. **Monitor with Azure Monitor**: Track API usage, errors, and latency
6. **Delete results when done**: Call the Delete Analyze Result API after retrieving results to clean up

In [ ]:
# Example: Delete analysis results after processing (data hygiene)
#
# from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
#
# poller = client.begin_analyze_document(
#     "prebuilt-invoice",
#     AnalyzeDocumentRequest(url_source=invoice_url)
# )
# result = poller.result()
# operation_id = poller.details["operation_id"]
#
# # ... process the result ...
#
# # Delete the analysis result from Azure (optional, auto-deletes after 24h)
# client.delete_analyze_result(model_id=result.model_id, result_id=operation_id)
# print(f"Analysis result {operation_id} deleted.")

print("Tip: Analysis results auto-delete after 24 hours.")
print("Use delete_analyze_result() for immediate cleanup of sensitive insurance data.")

---

## Key Takeaways

| Pattern | Best For |
|---|---|
| **Sequential with retry** | Simple workflows, small batches |
| **Concurrent submission** | Higher throughput, medium batches |
| **Directory processing** | Bulk digitization, migration projects |

| Best Practice | Why |
|---|---|
| Exponential backoff | Graceful handling of rate limits |
| Delay between submissions | Stay under TPS limits proactively |
| Delete after processing | Data hygiene for sensitive insurance data |
| Request TPS increase | Scale to production volume |

## Next Steps

- Read [Resilience & DR](../docs/resilience-and-dr.md) for cross-region failover
- Read [Security & Compliance](../docs/security-privacy-compliance.md) for data protection
- See [Resources](../docs/resources.md) for SDK samples and documentation